# 02 · Preprocessing & metadata extraction (v1.1)
Group 6 (Sparse / BM25 RAG) · Phase 1 · Member 2 – Preprocessing & Metadata

This notebook (1) loads the frozen splits, (2) cleans and tokenises the captions, (3) removes the knowledge-base captions that leak into test/val (release v1.1), (4) detects objects directly in the images with YOLOv8m, and (5) checks the outputs.

| Step | Output |
|---|---|
| Cleaned captions (v1.1) | `captions_processed_v1.1.json` |
| Objects detected in the images | `metadata.json` |
| Summary statistics | `preprocessing_stats.json` |

**How to run:** on Kaggle with a **GPU** (Tesla T4), add the COCO 2017 dataset and the dataset containing the v1.1 splits (`kb_manifest.json`, `val_manifest.json`, `test_manifest.json` **and `excluded_captions.json`**) as inputs, then *Restart & Run All*. Paths can be overridden with the environment variables `SPLITS_DIR`, `DATA_ROOT` and `OUT_DIR`.

## 0. Configuration

In [ ]:
import os, re, json, time, collections
from pathlib import Path

# ---- Paths (override with environment variables) ----
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/kaggle/input/datasets/sabahesaraki/2017-2017"))
OUT_DIR   = Path(os.environ.get("OUT_DIR", "/kaggle/working"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

def find_splits_dir():
    """Folder that contains kb_manifest.json (SPLITS_DIR, or searched under /kaggle/input)."""
    if os.environ.get("SPLITS_DIR"):
        return Path(os.environ["SPLITS_DIR"])
    for root, _, files in os.walk("/kaggle/input"):
        if "kb_manifest.json" in files:
            return Path(root)
    raise FileNotFoundError("kb_manifest.json not found: add the splits dataset as input or set SPLITS_DIR")

SPLITS_DIR = find_splits_dir()
print("splits folder:", SPLITS_DIR)

# ---- Parameters ----
CONF = 0.25          # detector confidence threshold (objects are predictions, not human labels)
EXCLUDED_FILE = SPLITS_DIR / "excluded_captions.json"

## 1. Load the frozen splits
The three manifests come from the frozen release (kb = 10,000, val = 100, test = 1,000). Image IDs must be disjoint.

In [ ]:
kb   = json.load(open(SPLITS_DIR / "kb_manifest.json"))
val  = json.load(open(SPLITS_DIR / "val_manifest.json"))
test = json.load(open(SPLITS_DIR / "test_manifest.json"))

ids = {"kb": {x["image_id"] for x in kb}, "val": {x["image_id"] for x in val}, "test": {x["image_id"] for x in test}}
assert (len(kb), len(val), len(test)) == (10_000, 100, 1_000)
assert not (ids["kb"] & ids["val"]) and not (ids["kb"] & ids["test"]) and not (ids["val"] & ids["test"])
print(f"KB={len(kb)} | Val={len(val)} | Test={len(test)} | TOTAL={len(kb)+len(val)+len(test)}")

## 2. Caption cleaning
Rules, applied identically to every split:
1. lowercase;
2. replace every character that is not a–z, 0–9, a space or an apostrophe by a space (punctuation removed; accented characters act as separators);
3. collapse repeated whitespace and strip the ends;
4. split on whitespace to get the tokens.

No stopword removal, stemming or lemmatisation is applied here. `captions_raw` is always kept.

In [ ]:
def clean(c):
    c = re.sub(r"[^a-z0-9\s']", " ", c.lower())
    return re.sub(r"\s+", " ", c).strip()

def build(items, split):
    return [{
        "image_id": x["image_id"],
        "split": split,
        "captions_raw":   x["captions"],
        "captions_clean": [clean(c) for c in x["captions"]],
        "tokens":         [clean(c).split() for c in x["captions"]],
    } for x in items]

caps = build(kb, "kb") + build(val, "val") + build(test, "test")
assert all(c for r in caps for c in r["captions_clean"]), "empty cleaned caption"
print(f"{len(caps)} entries, {sum(len(r['captions_clean']) for r in caps)} cleaned captions")

## 3. Caption exclusion (release v1.1)
The caption-level leakage audit (Member 1) found knowledge-base captions that are identical, after normalisation, to test or validation captions. They are listed in `excluded_captions.json` and are removed from `captions_clean` and `tokens` of the **knowledge base only**. `captions_raw`, the images, and the test/val captions are not changed.

The notebook stops with an error if the exclusion file is missing, so the exclusion can never be skipped silently.

In [ ]:
assert EXCLUDED_FILE.exists(), f"{EXCLUDED_FILE} not found: the v1.1 exclusion cannot be skipped"
excluded = json.load(open(EXCLUDED_FILE))
drop = {(e["kb_image_id"], e["caption"].strip()) for e in excluded}
print("exclusion rules:", len(drop))

removed = 0
kept_raw = {}     # only used for the leakage check below
for r in caps:
    if r["split"] != "kb":
        continue
    keep = [(r["image_id"], c.strip()) not in drop for c in r["captions_raw"]]
    removed += keep.count(False)
    r["captions_clean"] = [c for c, k in zip(r["captions_clean"], keep) if k]
    r["tokens"]         = [t for t, k in zip(r["tokens"], keep) if k]
    kept_raw[r["image_id"]] = [c for c, k in zip(r["captions_raw"], keep) if k]

kb_clean = sum(len(r["captions_clean"]) for r in caps if r["split"] == "kb")
print(f"captions removed: {removed} | KB cleaned captions left: {kb_clean}")
assert removed == len(drop) == 223 and kb_clean == 49_777

# check: no remaining KB caption equals a test/val caption (lowercase, punctuation removed)
norm = lambda s: " ".join(re.sub(r"[^\w\s]", " ", s.lower()).split())
ref = {norm(c) for x in val + test for c in x["captions"]}
leak = sum(norm(c) in ref for cs in kept_raw.values() for c in cs)
print("KB captions still equal to a test/val caption:", leak)
assert leak == 0

json.dump(caps, open(OUT_DIR / "captions_processed_v1.1.json", "w"), indent=2)
print("saved captions_processed_v1.1.json")

## 4. Object extraction from the images
Objects are detected **directly in the images** with YOLOv8m (pretrained COCO weights), one image at a time, with a confidence threshold of 0.25. Each image keeps its own objects (label, confidence, bounding box `[x_min, y_min, x_max, y_max]` in pixels).

The threshold is 0.25 and not 1 because the objects are predictions of a detector; a threshold of 1 only applies to metadata taken from the human-written captions.

In [ ]:
!pip -q install ultralytics
import ultralytics, torch
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else "cpu"
print("device:", "GPU " + torch.cuda.get_device_name(0) if device == 0 else "CPU (very slow, use a GPU)")
model = YOLO("yolov8m.pt")

def get_path(x, split):
    folder = "train2017" if split in ("kb", "val") else "val2017"
    p1 = DATA_ROOT / folder / folder / x["file_name"]
    return p1 if p1.exists() else DATA_ROOT / folder / x["file_name"]

items = [(x, "kb") for x in kb] + [(x, "val") for x in val] + [(x, "test") for x in test]
assert get_path(*items[0]).exists(), "image folder not found: check DATA_ROOT"

In [ ]:
from tqdm.auto import tqdm

meta = []
t0 = time.time()
for x, s in tqdm(items, desc="YOLOv8m"):
    r = model.predict(str(get_path(x, s)), conf=CONF, verbose=False, device=device)[0]
    objs = [{
        "label": r.names[int(cls)],
        "conf":  round(float(score), 3),
        "bbox":  [round(float(v), 1) for v in box],
    } for cls, score, box in zip(r.boxes.cls, r.boxes.conf, r.boxes.xyxy)]
    meta.append({"image_id": x["image_id"], "split": s, "objects": objs})
elapsed = time.time() - t0
print(f"{len(meta)} images processed in {elapsed/60:.1f} min")

json.dump({
    "detector": "YOLOv8m",
    "ultralytics_version": ultralytics.__version__,
    "conf_threshold": CONF,
    "data": meta,
}, open(OUT_DIR / "metadata.json", "w"), indent=2)
print("saved metadata.json")

## 5. Quality checks and statistics
Checks: every image has exactly one entry that matches the split IDs, confidences respect the threshold, captions and tokens stay aligned. The statistics are saved to `preprocessing_stats.json` for the report.

In [ ]:
import statistics as st

assert len(meta) == 11_100, f"{len(meta)} images instead of 11,100"
for s in ("kb", "val", "test"):
    got = [m["image_id"] for m in meta if m["split"] == s]
    assert len(got) == len(set(got)) and set(got) == ids[s], f"metadata does not match split {s}"
assert all(o["conf"] >= CONF for m in meta for o in m["objects"])
assert all(len(r["captions_clean"]) == len(r["tokens"]) for r in caps)

stats = {"detector": "YOLOv8m", "ultralytics_version": ultralytics.__version__, "conf_threshold": CONF,
         "detection_minutes": round(elapsed / 60, 1), "splits": {}}
for s in ("kb", "val", "test"):
    n = [len(m["objects"]) for m in meta if m["split"] == s]
    cc = [r for r in caps if r["split"] == s]
    stats["splits"][s] = {
        "images": len(n), "objects": sum(n), "mean_objects": round(sum(n) / len(n), 2),
        "median_objects": st.median(n), "max_objects": max(n), "images_without_objects": sum(1 for x in n if x == 0),
        "cleaned_captions": sum(len(r["captions_clean"]) for r in cc),
    }
all_obj = [o for m in meta for o in m["objects"]]
stats["total_objects"] = len(all_obj)
stats["images_without_objects"] = sum(1 for m in meta if not m["objects"])
stats["mean_objects_per_image"] = round(len(all_obj) / len(meta), 2)
stats["distinct_classes"] = len({o["label"] for o in all_obj})
stats["top_classes"] = collections.Counter(o["label"] for o in all_obj).most_common(10)
tok_len = [len(t) for r in caps for t in r["tokens"]]
stats["tokens_per_caption"] = {"mean": round(sum(tok_len) / len(tok_len), 2), "median": st.median(tok_len),
                               "min": min(tok_len), "max": max(tok_len)}

json.dump(stats, open(OUT_DIR / "preprocessing_stats.json", "w"), indent=2)
print(json.dumps(stats, indent=2))
print("\nAll checks passed.")